# Bigtable

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Google Cloud Bigtable is a wide-column NoSQL store for petabyte-scale, low-latency workloads such as time series, IoT and user profiles. It serves sparse rows through column families and splits data across tablets per node, so key and schema design decide performance. The questions below cover the data model, access patterns, replication and operations. The repetitive capability checklist from generic interview lists has been collapsed to one answer per topic, and the *Things to remember* sections near the end compress the limits and mistakes worth memorizing.


## Contents

<details>
<summary>46 sections - click to expand</summary>

- [What is Bigtable?](#what-is-bigtable)
- [How do you design a good Bigtable row key?](#how-do-you-design-a-good-bigtable-row-key)
- [When should you use Bigtable instead of BigQuery?](#when-should-you-use-bigtable-instead-of-bigquery)
- [How does Bigtable handle data storage?](#how-does-bigtable-handle-data-storage)
- [What is a tablet in Bigtable?](#what-is-a-tablet-in-bigtable)
- [How does Bigtable ensure fault tolerance?](#how-does-bigtable-ensure-fault-tolerance)
- [How does Bigtable handle load balancing?](#how-does-bigtable-handle-load-balancing)
- [What consistency model does Bigtable provide?](#what-consistency-model-does-bigtable-provide)
- [What are some typical use cases for Bigtable?](#what-are-some-typical-use-cases-for-bigtable)
- [How does Bigtable handle schema changes?](#how-does-bigtable-handle-schema-changes)
- [Can you explain the architecture of Bigtable?](#can-you-explain-the-architecture-of-bigtable)
- [How does Bigtable handle data replication?](#how-does-bigtable-handle-data-replication)
- [Does Bigtable support ACID transactions?](#does-bigtable-support-acid-transactions)
- [How can you interact with Bigtable?](#how-can-you-interact-with-bigtable)
- [What is the recommended way to perform atomic row-level updates in Bigtable?](#what-is-the-recommended-way-to-perform-atomic-row-level-updates-in-bigtable)
- [How does Bigtable handle access control and security?](#how-does-bigtable-handle-access-control-and-security)
- [Can you describe the data model used in Bigtable?](#can-you-describe-the-data-model-used-in-bigtable)
- [What is the maximum size of a row in Bigtable?](#what-is-the-maximum-size-of-a-row-in-bigtable)
- [How does Bigtable handle data compression?](#how-does-bigtable-handle-data-compression)
- [Does Bigtable support secondary indexes?](#does-bigtable-support-secondary-indexes)
- [How does Bigtable handle hotspots?](#how-does-bigtable-handle-hotspots)
- [How does Bigtable handle backups and disaster recovery?](#how-does-bigtable-handle-backups-and-disaster-recovery)
- [Can you explain the difference between Bigtable and HBase?](#can-you-explain-the-difference-between-bigtable-and-hbase)
- [What are the considerations for choosing between Bigtable and other databases like Cassandra or MongoDB?](#what-are-the-considerations-for-choosing-between-bigtable-and-other-databases-like-cassandra-or-mongodb)
- [How does Bigtable handle compaction?](#how-does-bigtable-handle-compaction)
- [Can you explain how Bigtable manages garbage collection?](#can-you-explain-how-bigtable-manages-garbage-collection)
- [How does Bigtable handle time-based data, such as event logs?](#how-does-bigtable-handle-time-based-data-such-as-event-logs)
- [Can you explain the role of a Bloom filter in Bigtable?](#can-you-explain-the-role-of-a-bloom-filter-in-bigtable)
- [Does Bigtable support automatic scaling of storage and compute resources?](#does-bigtable-support-automatic-scaling-of-storage-and-compute-resources)
- [How does Bigtable handle time-travel queries?](#how-does-bigtable-handle-time-travel-queries)
- [Does Bigtable support change data capture (CDC) for real-time data integration?](#does-bigtable-support-change-data-capture-cdc-for-real-time-data-integration)
- [Can you explain how Bigtable handles data versioning?](#can-you-explain-how-bigtable-handles-data-versioning)
- [What is the role of Bigtable's memstore?](#what-is-the-role-of-bigtables-memstore)
- [Does Bigtable provide automatic data expiration?](#does-bigtable-provide-automatic-data-expiration)
- [Does Bigtable provide support for aggregations and analytics?](#does-bigtable-provide-support-for-aggregations-and-analytics)
- [Can you explain how Bigtable handles storage and retrieval of large objects?](#can-you-explain-how-bigtable-handles-storage-and-retrieval-of-large-objects)
- [Can you explain how Bigtable handles range scans and filters?](#can-you-explain-how-bigtable-handles-range-scans-and-filters)
- [Can you explain how Bigtable handles large-scale data migration?](#can-you-explain-how-bigtable-handles-large-scale-data-migration)
- [Can you explain the role of Bigtable's read-modify-write operation?](#can-you-explain-the-role-of-bigtables-read-modify-write-operation)
- [Can you explain the role of Bigtable's client-side buffering and batching in optimizing write operations?](#can-you-explain-the-role-of-bigtables-client-side-buffering-and-batching-in-optimizing-write-operations)
- [Tuning row keys for even load and locality](#tuning-row-keys-for-even-load-and-locality)
- [Things to remember: key numbers and limits](#things-to-remember-key-numbers-and-limits)
- [Things to remember: semantics to memorize](#things-to-remember-semantics-to-memorize)
- [Things to remember: mistakes that keep coming back](#things-to-remember-mistakes-that-keep-coming-back)
- [Real world use cases for Bigtable](#real-world-use-cases-for-bigtable)
- [Bigtable exercises](#bigtable-exercises)

</details>

Tip: the links open the sections on GitHub; in Jupyter or Colab, use the notebook's outline panel in the sidebar instead.

## What is Bigtable?

Bigtable is a distributed, highly scalable, and NoSQL database developed by Google.

**Mental model.** Bigtable is a sparse, distributed, sorted map. A table maps one row key to one row, and a row holds column families, each with columns (qualifiers) and versioned cells. The map is sorted, so a range of row keys is one cheap scan.

**How it works.** Data is split into tablets (contiguous key ranges) served by a cluster of nodes. Nodes hold no durable data: SSTables live in Colossus, Google's distributed file system, so storage and compute scale and fail independently. This is the key difference from a single-node store.

**Gotcha.** Bigtable is not SQL and has no joins. If a query needs a join, an ad hoc `GROUP BY`, or a full-table aggregation, it belongs in BigQuery, not here.

Worked example: one table, one row key, four dimensions.

```text
row key:        acme#dev-00042#2026-10-04T12:00:00Z
column family:  r
column:         temperature
cell:           value=21.4, timestamp=2026-10-04T12:00:00Z
```

The same row key can carry `humidity`, `battery`, and later versions of `temperature`. Sparse means rows do not need the same columns, so the schema never has to be padded with nulls.

Common mistake: treating a table as relational and adding one row per column. Model one entity per row and put the varying fields in columns.

## How do you design a good Bigtable row key?

A good Bigtable row key supports the query pattern and avoids hotspotting. Bigtable stores rows lexicographically by key, so monotonically increasing keys like raw timestamps can concentrate writes on one tablet. Use prefixes, salting, bucketing, or reversed timestamps when needed.

Example:

```text
customer_id#reverse_timestamp
region#bucket#device_id#timestamp
```

In [ ]:
from datetime import datetime

MAX_TS = 9_999_999_999   # far-future sentinel for reversed timestamps

def row_key(tenant: str, device_id: str, ts: datetime) -> str:
    # salt spreads hot tenants across tablets; reversed timestamp sorts newest first
    bucket = hash((tenant, device_id)) % 16
    reverse_ts = MAX_TS - int(ts.timestamp())
    return f"{bucket:02d}#{tenant}#{device_id}#{reverse_ts:010d}"

The same idea with lexicographic prefixes:

```text
00#acme#dev-00042#9999999999   -> newest first inside one device
00#acme#dev-00042#9999999900
01#globex#dev-00007#9999999998
```

**Mental model.** The row key is the only index, so it must encode the access pattern. Rows sort lexicographically by raw bytes. A prefix scan reads every row whose key starts with a prefix, in sorted order, from one contiguous key range.

**How it works.** Put the field you filter on first, then the field you sort by, then a unique suffix. Keep keys short (tens of bytes); long keys cost storage and cache on every row. Use fixed-width, zero-padded numbers so byte order matches numeric order: `2026-10-04` sorts correctly, `10/4/2026` does not.

**Gotcha.** A key that starts with a monotonically increasing value (a timestamp, a sequence, a `user-00001` counter) sends all new writes to one tablet end. That is the classic hotspot.

Worked example: queries drive the key.

```text
Query: latest readings for one device, newest first.
Key:   acme#dev-00042#9999999999   (reversed timestamp for newest-first)
Query: all devices in a region over a time window.
Key:   region#device_id#ts         (region first, so a region is one scan)
```

Common mistake: choosing the key from the entity shape instead of the query list. Write down the top three reads and writes first, then design the key to make each a prefix scan or a point lookup.

## When should you use Bigtable instead of BigQuery?

Use Bigtable for low-latency operational lookups, time-series access, and high-throughput key-value style workloads. Use BigQuery for analytical SQL, joins, scans, and aggregations. Bigtable is not a general data warehouse; it is optimized for serving rows by key/range.

Example:

+ Bigtable: fetch recent IoT readings for one device in milliseconds
+ BigQuery: calculate monthly usage by region across all devices

Many architectures use both: Bigtable for serving and BigQuery for analytics.

**Mental model.** Bigtable serves individual rows and narrow key ranges at low latency and high request rates. BigQuery scans columns across billions of rows for analytics. They answer different questions: "what is this user's state now" versus "what did all users do last quarter".

**How it works.** A Bigtable read is an index seek plus a few disk or cache reads, usually single-digit milliseconds. A BigQuery query reads columnar files in parallel and is billed by bytes scanned, so it prefers large scans and dislikes point lookups.

**Gotcha.** Running analytics on Bigtable forces full-table scans through a serving path and crowds out application traffic. Keep them apart, or read analytics through BigQuery federation or Data Boost.

Worked example: the same event stream, two serving needs.

```text
Serving:  get latest 20 events for user u-7781        -> Bigtable point + range read
Analytics: p95 latency by region for the last 90 days -> BigQuery scan
```

Common mistake: putting a hot dashboard on Bigtable "because the data is already there". Dashboards scan many rows and belong on a columnar copy.

Many architectures use both: Bigtable for serving, BigQuery for analytics, Change Streams or Dataflow to keep the copy fresh.

## How does Bigtable handle data storage?

Bigtable stores data in a sparse, distributed, and multi-dimensional sorted map.

**Mental model.** Bigtable stores a sparse, distributed, multidimensional sorted map. Each cell is `(row key, column family, column qualifier, timestamp) -> bytes`. Unset cells cost nothing, which is why wide and sparse rows are normal.

**How it works.** A write lands in a tablet node's commit log and in-memory memtable. When the memtable fills it flushes to an immutable SSTable in Colossus. A compaction later merges SSTables and drops versions the GC policy rejects. Nodes serve from cache, memtable, and SSTables; durable bytes always live in Colossus.

**Gotcha.** Storage is decoupled from nodes, so adding nodes does not add storage in the same way it would on a local-disk system. Size the cluster for throughput, and let storage grow independently.

Worked example: the write path in order.

```text
1. client sends mutation to the tablet node that owns the key range
2. node appends to the commit log (durability) and the memtable (readable now)
3. memtable flush -> SSTable in Colossus
4. compaction merges SSTables and applies the GC policy
```

Common mistake: assuming a delete frees space immediately. A delete writes a tombstone, and the bytes go away only when compaction removes the tombstoned versions.

## What is a tablet in Bigtable?

A tablet is a range of rows in a Bigtable that is stored and managed independently by a single server.

**Mental model.** A tablet is a contiguous range of row keys and the unit of distribution and load balancing, roughly the region of HBase. A table is a set of non-overlapping tablets that together cover the whole key space.

**How it works.** The service splits a tablet as it grows and merges neighbours when they shrink, so the number of tablets tracks data size rather than a fixed shard count. Tablets move between nodes as load changes, and clients are told which node serves which range.

**Gotcha.** The number of tablets is not something you set. You cannot pre-split or manually split a tablet in Cloud Bigtable, so a key that creates one huge range is fixed only by changing the key.

Worked example: key range to tablets.

```text
acme#dev-00001 .. acme#dev-09999  -> tablet A
acme#dev-10000 .. acme#dev-24999  -> tablet B
globex#...                        -> tablet C
```

Common mistake: assuming one tablet per table or one per node. The service decides, and it rebalances as the data grows.

## How does Bigtable ensure fault tolerance?

Bigtable maintains multiple replicas of each tablet, ensuring data durability and availability in case of server failures.

**Mental model.** Durable data lives in Colossus with replication, so a dead node loses nothing. Compute is stateless, so the service reassigns its tablets to healthy nodes. A multi-cluster instance adds another copy in a second zone or region.

**How it works.** The commit log and SSTables are replicated by the storage layer. Bigtable detects an unhealthy node, moves its tablets, and clients retry against the new location. Replication between clusters is asynchronous and eventual.

**Gotcha.** Fault tolerance is not backup. A bad application write or a dropped table replicates to every cluster in milliseconds; only a backup restores it.

Worked example: two failure types, two protections.

```text
node or zone failure -> automatic, service-managed (replication + tablet reassignment)
bad deploy, dropped table -> restore from a backup into a new table
```

Common mistake: treating multi-cluster replication as a restore point. It copies mistakes, not history.

## How does Bigtable handle load balancing?

Bigtable automatically redistributes tablets across servers to balance the workload and maintain performance.

**Mental model.** The service moves tablets between nodes so no node serves too many hot ranges. Balancing reacts to load signals, not just to byte counts, so a hot tablet is spread even if the table's data is uneven.

**How it works.** As tablets split, the new tablets are placed on different nodes. Nodes that are overloaded offload tablets; underloaded nodes pick them up. Replication and locality constraints (for example keeping data near an app profile's cluster) bound where a tablet may go.

**Gotcha.** Rebalancing cannot split a single hot row. If every request uses the same row key, all of it stays on one tablet on one node.

Worked example: balancing versus key design.

```text
even key spread + hot node   -> service rebalances and fixes it
one hot row key              -> nothing to rebalance; change the key
```

Common mistake: watching node CPU and adding nodes when the real problem is one hot row. Check the per-tablet and per-prefix metrics first.

## What consistency model does Bigtable provide?

Within a single cluster, Bigtable provides strong consistency: a read sees the result of every completed write. When an instance has clusters in more than one region, replication between them is asynchronous, so data converges across clusters instead of being immediately consistent. App profile routing decides which behavior a client gets: single-cluster routing for strict consistency, multi-cluster routing for lower-latency replicated reads.

**Mental model.** Within a single cluster, reads are strongly consistent: after a write commits, every later read sees it. Across clusters, replication is asynchronous, so a read from another cluster may briefly see an older version.

**How it works.** A normal app profile uses single-cluster routing and reads and writes the same cluster, so consistency is immediate. A multi-cluster app profile can route reads anywhere for lower latency, and accepts eventual consistency in exchange.

**Gotcha.** Do not mix the two expectations. If the application needs read-your-writes, pin it to one cluster, or read the row back from the cluster that accepted the write.

Worked example: choosing routing by need.

```text
read-your-writes required      -> single-cluster app profile
nearest-region reads required  -> multi-cluster app profile, eventual consistency
```

Common mistake: enabling multi-cluster routing to cut latency, then filing a bug that a just-written value is missing. That is the documented trade-off, not a fault.

## What are some typical use cases for Bigtable?

Bigtable is commonly used for storing and analyzing large amounts of time-series data, as well as for serving real-time applications.

**Mental model.** The fit is a large number of entities, each read or written by key or by a narrow ordered range, at high request rates with low and predictable latency. If the access is a scan and aggregate, it is a warehouse job.

**How it works.** Common shapes are time-series telemetry, per-user state and message history, feature serving, real-time bidding profiles, personalization, graph or index stores, and recommendation inputs. All share a key-first access pattern.

**Gotcha.** Bigtable is a poor fit for multi-row transactions, ad hoc joins, and full-text relevance ranking. Use a relational store, a warehouse, or a search engine for those.

Worked example: matching query to system.

```text
latest reading for a device         -> Bigtable
sum of usage by region for a month  -> BigQuery
full-text search over messages      -> a search engine, fed from Bigtable
```

Common mistake: forcing every access pattern into one store. Most production systems pair Bigtable with a warehouse and a search index.

## How does Bigtable handle schema changes?

Bigtable is schema-less, meaning that new columns can be added to the table without affecting existing rows.

**Mental model.** There is almost no schema to migrate. Adding a column family is an online admin operation, and adding a new column qualifier is just writing it. Existing rows simply do not have the new cell, which is what sparse means.

**How it works.** `cbt createfamily` or the client library adds a family without downtime. Removing a family drops its data. New qualifiers appear the first time a writer uses them, and readers must tolerate a missing cell.

**Gotcha.** Changing a GC policy is not retroactive in the way people expect; it takes effect as data is compacted, so old versions can linger briefly after you tighten the rule.

Worked example: adding a field with no migration.

```bash
cbt -project acme-prod -instance acme-bt createfamily device_readings battery
# writers start setting r:battery; older rows return no cell for it
```

Common mistake: writing migration jobs to backfill a new column. Read code should treat a missing cell as a default, and backfill only if analytics needs it.

## Can you explain the architecture of Bigtable?

Bigtable has a distributed architecture with multiple components, including tablets, tablet servers, and a master server for coordination.

**Mental model.** Four moving parts: clients, a metadata layer that maps key ranges to nodes, the tablet nodes that serve reads and writes, and Colossus that stores the SSTables and commit logs. Data and compute are separated.

**How it works.** A client asks which node owns a key, caches the answer, and sends the request directly to that node. The node reads cache, memtable, or SSTables and returns rows. The service splits, merges, and moves tablets in the background.

**Gotcha.** The row key is the shard key. Architecture diagrams often show even sharding, but uneven key prefixes produce uneven tablets regardless of how many nodes exist.

Worked example: request path for a point read.

```text
client -> metadata lookup (cached) -> tablet node for the key range
       -> memtable/cache/SSTable -> row bytes -> client
```

Common mistake: drawing Bigtable as a single primary with replicas for scale. There is no single primary for data; nodes are peers serving different key ranges.

## How does Bigtable handle data replication?

Bigtable replicates data across multiple data centers to ensure durability and availability in case of failures.

**Mental model.** Replication is per instance and per cluster. You add a second cluster in another zone or region, and Bigtable copies writes to it asynchronously. Each cluster holds a full copy of the table.

**How it works.** A write goes to one cluster first and is then replicated to the others. An app profile decides where a client reads and writes: one cluster for strong consistency, or any cluster for low-latency reads with eventual consistency. You can also route by cluster group to keep some data regional.

**Gotcha.** Replication is not a transaction across clusters. There is no cross-cluster atomic commit, and conflicting concurrent writes to the same cell resolve by last-write-wins on the server timestamp.

Worked example: two clusters, two app profiles.

```text
instance acme-bt
  cluster acme-bt-us-east1   (us-east1)
  cluster acme-bt-eu-west1   (europe-west1)
app profile "serving"  -> multi-cluster routing, reads nearest
app profile "billing"  -> single-cluster acme-bt-us-east1, read-your-writes
```

Common mistake: assuming a failover promotes a primary. There is no primary; clients reroute through the app profile.

## Does Bigtable support ACID transactions?

Bigtable gives ACID semantics at the row level: a row mutation, including a conditional read-modify-write, is atomic. Cross-row transactions are not the design center, so model around single-row atomicity instead of expecting relational-style multi-row transactions.

**Mental model.** Bigtable supports atomicity at the row level, not across rows. A single row mutation, which may touch many columns, applies as one unit. Conditional mutations give you compare-and-set on a row.

**How it works.** Use `CheckAndMutateRow` (Python: `row.commit()` after a conditional row, or the `conditional_row` API) to apply changes only if a predicate holds. This supports read-modify-write and optimistic concurrency within one row.

**Gotcha.** A batch that spans rows is not atomic. If one mutation in a batched write fails, others may already have applied, so retry logic must be idempotent.

Worked example: atomic counter update on one row.

```python
row = table.conditional_row(b"acme#counter")
row.set_cell("c", b"value", b"1")
if_true = row_filters.ValueRegexFilter(b"^0$")
row.commit()  # applies only when the predicate matches
```

Common mistake: moving money between two rows in one operation. Do it as two idempotent single-row mutations with a reconciliation step, or keep that data in a transactional database.

## How can you interact with Bigtable?

Bigtable provides client libraries for different programming languages, such as Java, Python, and Go, to interact with the database.

**Mental model.** Four client paths: the `cbt` command line for admin and debugging, the language client libraries for applications, the GoogleSQL surface for row-key queries, and BigQuery federation for analytics. All talk to the same service.

**How it works.** `cbt` is best for creating tables, families, and GC policies and for reading a few rows. The Python, Java, Go, and other clients give connection pooling, batching, and retries. The HBase client library works against Bigtable for HBase migrations.

**Gotcha.** `cbt` is not a production driver. It opens a connection per command and has no pooling, so use it for operations and use a client library for application traffic.

Worked example: the everyday commands.

```bash
cbt -project acme-prod -instance acme-bt ls
cbt -project acme-prod -instance acme-bt ls device_readings
cbt -project acme-prod -instance acme-bt read device_readings prefix=acme#
cbt -project acme-prod -instance acme-bt count device_readings
```

Common mistake: scripting `cbt` in a hot path. It is a debugging and admin tool, not a low-latency client.

## What is the recommended way to perform atomic row-level updates in Bigtable?

Bigtable supports atomic row-level updates using conditional mutations, allowing you to update multiple cells within a row atomically.

**Mental model.** Treat a row as the unit of consistency. Read the row, decide, and write it back with a conditional mutation so a concurrent writer cannot interleave. If the condition fails, re-read and retry.

**How it works.** `CheckAndMutateRow` takes a predicate filter and two mutation sets, one for true and one for false, and applies exactly one of them atomically. The read and the write are not a transaction, so you use the condition to detect interference and retry.

**Gotcha.** An unconditional read-modify-write is a lost-update bug under concurrency. Always guard it with a predicate, or use an aggregate family for counters.

Worked example: optimistic concurrency with a version cell.

```python
row = table.conditional_row(b"acct-7781")
row.set_cell("s", b"balance", new_balance)
row.set_cell("s", b"version", str(version + 1).encode())
row.set_cell("s", b"pending", b"1", state=False)  # clear only on success
row.commit()
```

Common mistake: reading outside the conditional and assuming the value is still current. Put the expected version in the predicate, not in application memory alone.

## How does Bigtable handle access control and security?

Bigtable integrates with Google Cloud IAM (Identity and Access Management) to provide fine-grained access control and security policies.

**Mental model.** Security has three layers: IAM roles for who may call which API, encryption for data at rest and in transit, and audit logging for who did what. There is no row-level SQL grant; access is per table through IAM.

**How it works.** Roles like `roles/bigtable.reader`, `roles/bigtable.user`, and `roles/bigtable.admin` are granted at the project, instance, or table level. Data is encrypted at rest by default, optionally with CMEK, and in transit with TLS. Data Access audit logs record reads and writes but are off by default.

**Gotcha.** IAM is not row-level. If two tenants must not see each other's rows, use separate tables or instances, or authorized views, not a column filter in the client.

Worked example: least privilege for a service account.

```bash
gcloud bigtable instances add-iam-policy-binding acme-bt \
  --member="serviceAccount:reader@acme.iam.gserviceaccount.com" \
  --role="roles/bigtable.reader"
```

Common mistake: granting project-wide `bigtable.admin` to an application so it can read one table. Grant the narrowest role at the narrowest resource.

## Can you describe the data model used in Bigtable?

Bigtable uses a sparse, distributed, and multidimensional sorted map, where data is indexed by a row key, column key, and timestamp.

**Mental model.** A table is a sorted map: row key to row. A row contains column families, a family contains columns (qualifiers), and a column contains cells, each a version stamped with a timestamp. The full address of a value is the tuple `(row, family, qualifier, timestamp)`.

**How it works.** Rows sort by row key bytes. Qualifiers sort within a family. Versions sort newest timestamp first. Timestamps are 64-bit values, usually microseconds; the server sets them by default, or you supply your own for import and replay.

**Gotcha.** The row key is the only automatic index. Reading by any other field means a full scan or a second table that you maintain as a secondary index.

Worked example: one address, three versions.

```text
table:         device_readings
row key:       acme#dev-00042#2026-10-04T12:00:00Z
family:        r
qualifier:     temperature
cells:         (ts=12:00:00, "21.4") (ts=11:59:00, "21.3") (ts=11:58:00, "21.1")
```

Common mistake: assuming a family is a column. A family is a container with one GC policy; the columns inside it are created on write and are not declared.

In [ ]:
# ILLUSTRATIVE - requires google: install to run, or read on.
from google.cloud import bigtable

# review-only: requires the google-cloud-bigtable package and a real instance
client = bigtable.Client(project="acme-prod", admin=True)
table = client.instance("acme-bt").table("device_readings")

row = table.direct_row(b"acme#dev-00042#2026-10-04T12:00:00Z")
row.set_cell("r", b"temperature", b"21.4", timestamp_micros=1_759_579_200_000_000)
row.set_cell("r", b"humidity", b"48")
row.commit()


## What is the maximum size of a row in Bigtable?

Bigtable enforces hard system limits on row key size, row size, and cell size (see the Cloud Bigtable quotas and limits page), and oversized rows hurt latency and write amplification. Design many small rows rather than a few huge ones, and keep large blobs in Cloud Storage with a reference in Bigtable.

**Mental model.** A row key is limited to 4 KiB, and the practical limit on a row is much smaller than a relational row. Bigtable is designed for many small rows, not a few huge ones.

**How it works.** A single cell value has a service maximum on the order of 100 MiB, but that ceiling is for rare blobs, not for routine data. Wide rows with thousands of columns can be written, but reading them is slow and they pin the tablet serving that key.

**Gotcha.** A row that grows without bound, such as one row per user holding all their events, becomes a hotspot and a slow read. Split time-ordered data across many rows.

Worked example: capping the row.

```text
bad:  row key = user-7781, qualifier = event#evt-0001  (unbounded row, one per user)
good: row key = user-7781#9999999999, qualifier = event  (one row per event)
```

Common mistake: storing a list in one cell and appending to it. Each append rewrites the whole value and creates a version; store one event per row instead.

## How does Bigtable handle data compression?

Bigtable provides built-in support for data compression, allowing you to save storage space and improve read and write performance.

**Mental model.** Bigtable compresses data automatically. Compression is not a table or column-family setting you choose, and the service picks algorithms per workload.

**How it works.** SSTables are compressed in Colossus, which cuts storage cost and the bytes read per seek. Similar values next to each other in key order compress better, so a well-clustered key layout also improves the compression ratio.

**Gotcha.** Client-visible payloads are not compressed by the service, so a huge cell still travels over the wire at its raw size. Compress large blobs in the application before storing them.

Worked example: what compression does and does not cover.

```text
storage in Colossus   -> compressed automatically
read bytes per seek   -> reduced by compression
wire payload to client-> not compressed by Bigtable
```

Common mistake: assuming Snappy is a knob to set. On Cloud Bigtable you do not pick the algorithm; on HBase you do, which is one of the differences between the two.

## Does Bigtable support secondary indexes?

There is no classic relational secondary index, and the traditional answer was to duplicate data under a second row-key pattern. Current releases add asynchronous secondary indexes and continuous materialized views, plus GoogleSQL for Bigtable, so less of that duplication has to be hand-built.

**Mental model.** There is no built-in secondary index. The row key is the only ordered index. Any other lookup path is a second table that you keep in sync, or a BigQuery copy.

**How it works.** The standard pattern is an index table whose rows are `indexed_value#primary_key` and whose cells point back to the main row. Writes update both rows, ideally with Change Streams and Dataflow rather than a fragile dual write. Continuous materialized views and asynchronous secondary indexes are also available for some patterns.

**Gotcha.** An index table must handle the same hotspot rules as any table. Indexing a low-cardinality field, like `status`, puts millions of rows under a few prefixes.

Worked example: index by email.

```text
main table:  row = u-7781            family profile: email = "ada@acme.com"
index table: row = ada@acme.com#u-7781   family i: pk = "u-7781"
lookup:      read index row, then read main row
```

Common mistake: writing the index inline with the main write and ignoring partial failure. Use Change Streams or a conditional reservation so the two stay consistent.

## How does Bigtable handle hotspots?

Splitting helps, but a monotonically increasing row key still funnels writes into one tablet until it splits, so salting or bucketing the key prefix is the real fix. Watch per-tablet load and CPU utilization in Cloud Monitoring, not just table-level throughput.

**Mental model.** A hotspot is a row key, or a small key range, that receives far more traffic than the rest. It happens for two reasons: sequential keys (timestamps, counters) concentrate writes at one end, and a single popular entity concentrates all reads on one row.

**How it works.** Because rows sort by key, appending `2026-10-04T12:00:00Z` sends every new row to the highest key. A famous user or a global config row creates a single-tablet read hotspot. Key Visualizer shows the hot ranges as bright bands over time.

**Gotcha.** Adding nodes does not fix a hotspot; the hot range still lives on one tablet. Fix the key: promote a high-cardinality field to the front, add a hash or salt prefix, or reverse the timestamp.

Worked example: two fixes.

```text
write hotspot: 2026-10-04T12:00:00Z#device-42
  fix:         42#device-42#9999999999   (salt prefix spreads writes)
read hotspot:  CONFIG#global-flags
  fix:         01#CONFIG#global-flags  (16 buckets) + a cache in the app
```

Common mistake: salting the middle of the key. The salt must be the first bytes, or it does not change the sort order or the tablet boundaries.

## How does Bigtable handle backups and disaster recovery?

Bigtable provides built-in backup and restore functionality, allowing you to create backups and restore data to a specific point in time.

**Mental model.** A backup is a consistent, independent copy of a table that you create on demand and restore into a new table. Replication copies current state; a backup preserves a point in time that replication has already overwritten.

**How it works.** A backup is stored in a cluster and carries an optional expiration; by default it does not expire, and you delete it yourself. Restore creates a new table, then you point the application at it. You can also copy a backup to another region for cross-region recovery.

**Gotcha.** A backup is not continuous. Between two backups, a bad write or a dropped table loses everything since the last one, so schedule backups against your recovery point objective.

Worked example: create and restore.

```bash
gcloud bigtable backups create acme-daily-2026-10-04 \
  --instance=acme-bt --cluster=acme-bt-us-east1 --table=device_readings
gcloud bigtable tables restore device_readings_restored \
  --instance=acme-bt --cluster=acme-bt-us-east1 \
  --source=projects/acme-prod/instances/acme-bt/clusters/acme-bt-us-east1/backups/acme-daily-2026-10-04
```

Common mistake: restoring over the live table. Restore always targets a new table name; cut over deliberately.

## Can you explain the difference between Bigtable and HBase?

Bigtable and HBase are similar in many ways, as HBase was inspired by Bigtable. The main difference lies in their underlying infrastructure: Bigtable runs on Google's infrastructure, while HBase runs on top of the Hadoop ecosystem.

**Mental model.** Bigtable and HBase share a design lineage: the Bigtable paper inspired HBase, and both use a sorted map, tablets or regions, memtables, and SSTables or HFiles. Bigtable is the managed Google service; HBase is the open-source system you run yourself.

**How it works.** HBase runs a Master, RegionServers, ZooKeeper, and HDFS, and you tune block cache, Bloom filters, region pre-splits, and compaction. Bigtable runs on Colossus with stateless nodes, manages splitting and compaction for you, and exposes IAM, replication, and autoscaling as service features.

**Gotcha.** The APIs differ. HBase has coprocessors, filters, and a richer client API that Bigtable does not fully support; Bigtable has no ZooKeeper or HDFS to operate, which is the main reason teams move.

Worked example: operational surface.

```text
HBase:    you run Master, RegionServers, ZooKeeper, HDFS; you pre-split regions
Bigtable: you run no servers; you set nodes, app profiles, and GC policies
```

Common mistake: migrating an HBase key design with sequential timestamps unchanged. The hotspot behaviour is the same, so a schema review is part of the migration.

## What are the considerations for choosing between Bigtable and other databases like Cassandra or MongoDB?

The choice depends on factors such as data volume, query patterns, scalability requirements, and the need for tight integration with other Google Cloud services.

**Mental model.** Ask three questions: what is the access pattern, what consistency does the application require, and who operates the system. Cassandra and MongoDB are self-managed (or managed on another cloud); Bigtable is a managed Google service with a key-range model.

**How it works.** Bigtable and Cassandra both scale horizontally with a wide-column model, but Cassandra uses consistent hashing and tunable quorum consistency, while Bigtable uses sorted key ranges and single-cluster strong reads. MongoDB is a document store with secondary indexes and richer queries, which Bigtable lacks.

**Gotcha.** Choosing on features alone misses operations. If the team cannot run a Cassandra ring or a MongoDB replica set, a managed service changes the total cost more than any benchmark.

Worked example: fit by requirement.

```text
ordered range scans by key, managed      -> Bigtable
multi-cloud, tunable quorum, self-run    -> Cassandra
ad hoc queries and secondary indexes     -> MongoDB (or a relational store)
```

Common mistake: picking Bigtable for a document workload that needs many secondary indexes. Build those indexes explicitly or choose the document store.

## How does Bigtable handle compaction?

Bigtable performs compaction by periodically merging smaller sorted files into larger ones, reducing storage overhead and improving read performance.

**Mental model.** Compaction merges immutable SSTables into fewer, larger ones and applies the garbage collection policy in the same pass. It is background work that reclaims space and keeps reads fast by limiting the number of files a read must check.

**How it works.** A flush produces a new SSTable from the memtable. Minor compactions merge small SSTables; major compactions rewrite a larger set and drop expired or tombstoned versions. Because GC happens during compaction, a tightened policy takes effect gradually.

**Gotcha.** Compaction consumes node CPU and disk bandwidth. A burst of writes produces many flushes, and the resulting compaction work competes with serving traffic, so size nodes for headroom, not for the average.

Worked example: from memtable to compacted SSTables.

```text
write -> memtable -> flush -> sstable-1
write -> memtable -> flush -> sstable-2
                 compaction merges sstable-1 + sstable-2, applies GC -> sstable-3
```

Common mistake: expecting a delete to reduce storage immediately. The tombstone and the old value remain until compaction merges the files that hold them.

## Can you explain how Bigtable manages garbage collection?

Bigtable uses an automatic garbage collection process to reclaim disk space by removing older versions of data that are no longer needed.

**Mental model.** GC is per column family and rule-based. A cell is kept or dropped by the rule when compaction processes it. There are two primitives, `max_num_versions` and `max_age`, and two combinators, `union` and `intersection`.

**How it works.** `max_num_versions` keeps the newest N versions of each cell. `max_age` keeps cells newer than a duration. `union` keeps a cell if any rule keeps it; `intersection` keeps it only if every rule keeps it. You set the rule with `cbt setgcpolicy` or the client library, and `cbt ls device_readings` shows the current policy.

**Gotcha.** GC runs at compaction, not at write time, so an expired cell can still be read until compaction reaches it. Read code must filter by timestamp if it needs strict freshness.

Worked example: the rules as JSON.

```text
keep newest 3 versions:      {"max_num_versions": 3}
keep 30 days:                {"max_age": "2592000s"}
newest 3 OR 30 days:         {"union": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
newest 3 AND 30 days:        {"intersection": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
```

The same policies from the command line:

```bash
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxversions=3
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxage=30d
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxversions=3 or maxage=30d
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxversions=3 and maxage=30d
cbt -project acme-prod -instance acme-bt ls device_readings
```

Common mistake: using `union` when the goal is to cap storage. Union is the permissive rule; it keeps a cell if either rule keeps it. Use `intersection` to keep only cells that satisfy both.

## How does Bigtable handle time-based data, such as event logs?

Bigtable uses a timestamp associated with each cell, allowing you to store and query time-series data efficiently.

**Mental model.** Time is part of the key, not only a cell timestamp. Put the entity first and a reversed timestamp second so the newest events sort first and a prefix scan returns the latest N without reading history.

**How it works.** A key like `device-42#9999999999` gives newest-first ordering for one device. A GC policy of `max_age` then drops old events automatically, so storage stays bounded without a cleanup job. For queries across devices, put the grouping field first: `2026-10-04#device-42#ts`.

**Gotcha.** A raw ascending timestamp as the leading key field is the classic hotspot: every new event goes to the last tablet. Reverse it, salt it, or bucket by hour.

Worked example: latest-first scan.

```text
key = device-42#9999999990, device-42#9999999890, ...
scan prefix=device-42#, limit 10 -> newest ten rows, one range read
GC policy max_age=7d on the family -> older events expire at compaction
```

Common mistake: storing the timestamp only as the cell timestamp and keying by event id. Then "latest for this device" needs a full scan of the device's rows.

## Can you explain the role of a Bloom filter in Bigtable?

A Bloom filter is a probabilistic data structure used by Bigtable to reduce disk I/O by filtering out irrelevant data during read operations.

**Mental model.** A Bloom filter answers "this file definitely does not contain the key" cheaply, so reads skip SSTables that cannot hold the row. In HBase you choose filter types and tune them; in Cloud Bigtable the service manages this internally.

**How it works.** When a read looks for a row, Bigtable consults per-file metadata before touching the file. A negative answer skips a disk read, which matters when many SSTables cover a key range. The service decides the filter scope from the workload.

**Gotcha.** Bloom filters help point reads, not scans. A scan of many rows must read the data anyway, so a filter cannot make it cheaper.

Worked example: where the filter saves work.

```text
point read of device-42
  sstable-1: filter says no   -> skip
  sstable-2: filter says maybe -> read and find the row
```

Common mistake: hunting for a Bigtable Bloom filter setting. Unlike HBase, there is no `BLOOMFILTER` column-family attribute to set here.

## Does Bigtable support automatic scaling of storage and compute resources?

Yes, Bigtable can automatically scale storage and compute resources based on workload patterns and configuration settings.

**Main point.** Storage scales automatically with the data. Compute scales when you change the node count, and autoscaling can do that for you within a minimum and maximum.

**How it works.** SSDs and Colossus grow with the table with no disk to provision. Autoscaling watches CPU utilization and request rate and adds or removes nodes between your bounds, with a target utilization. Storage usage and node count are separate concerns and separate costs.

**Gotcha.** Autoscaling cannot fix a hotspot. If the load is concentrated on one key range, adding nodes does not spread it, and the cluster stays hot and expensive.

Worked example: an autoscaling policy.

```text
min nodes: 3   max nodes: 12
target CPU utilization: 60%
storage: grows with the table, independent of node count
```

Common mistake: setting the minimum to one node for a production table and relying on scale-up to absorb a traffic spike. Scaling is not instant, so budget headroom.

## How does Bigtable handle time-travel queries?

Bigtable allows you to retrieve previous versions of data by specifying a timestamp or a time range in your queries.

**Mental model.** There is no `AS OF` query and no system time-travel. Versioned cells let you read older values if they are still present, bounded by the column family's GC policy.

**How it works.** Each cell keeps multiple versions until compaction drops them. A read can ask for all versions or filter by a timestamp range, so you can reconstruct recent history. Backup and restore covers point-in-time recovery at table granularity.

**Gotcha.** History is not guaranteed. Once `max_num_versions` or `max_age` has dropped a version, it is gone. Never rely on a cell version as an audit log; write changes to a dedicated table or stream them out.

Worked example: reading older versions.

```python
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"device-42#",
    end_key=b"device-42$",
    filter_=row_filters.CellsColumnLimitFilter(5),  # newest five versions
)
```

Common mistake: promising "we can query any past state" from cell versions. Say instead that history is bounded by the GC policy, and use backups for recovery.

## Does Bigtable support change data capture (CDC) for real-time data integration?

Bigtable supports change data capture with Change Streams: you enable a change stream on a table and read mutation records from it, usually with Dataflow. Managed templates stream changes to BigQuery or Pub/Sub, which is the modern path for keeping a warehouse, cache, or downstream service in sync with Bigtable.

**Mental model.** Change Streams is Bigtable's CDC feature. You enable it on a table, and Bigtable appends a record of every row change to a changelog that a stream processor reads within seconds.

**How it works.** A change stream can capture the latest value or all values. A Dataflow or Beam job reads the stream and writes it to BigQuery, Pub/Sub, or another sink. The retention window for streamed records is configurable and capped at a few days, so consumers must keep up.

**Gotcha.** A change stream is not a full history and not a transaction log. Records can be delivered more than once, so the sink must be idempotent.

Worked example: serving plus analytics.

```text
device_write -> Bigtable tables/device_readings (change stream enabled)
             -> Dataflow reads change stream -> BigQuery analytics
```

Common mistake: dual-writing to Bigtable and BigQuery from the application and hoping they stay consistent. Let the change stream be the single path.

## Can you explain how Bigtable handles data versioning?

Bigtable assigns a unique timestamp to each cell, allowing multiple versions of a cell's data to be stored and retrieved.

**Mental model.** Every cell holds a list of timestamped versions, newest first. A write adds a version; it never overwrites history in place. A read returns the newest version unless you ask for more.

**How it works.** The server assigns the timestamp when you do not supply one, in microseconds. You can write your own timestamp for imports, replay, or out-of-order data. GC policy controls how many versions survive, and compaction enforces it.

**Gotcha.** A custom timestamp must respect the GC rules. If you import with old timestamps and the family uses `max_age`, the imported data can be garbage-collected almost immediately.

Worked example: version list on one cell.

```text
family r, qualifier temperature:
  ts 2026-10-04T12:00:00Z  "21.4"   <- read returns this
  ts 2026-10-04T11:59:00Z  "21.3"
  ts 2026-10-04T11:58:00Z  "21.1"
```

Common mistake: using a non-monotonic custom timestamp and then wondering why the newest value does not appear. Reads sort by timestamp descending.

## What is the role of Bigtable's memstore?

Bigtable's memstore is an in-memory data structure that temporarily holds recently written data before flushing it to disk.

**Mental model.** The memtable (memstore in HBase terms) is the in-memory write buffer for a tablet. Writes become readable as soon as they are in the memtable and the commit log, without waiting for a disk flush.

**How it works.** Each tablet node buffers writes for its tablets, then flushes to an SSTable when the buffer fills. A read checks the memtable first, then cache, then SSTables. The service sizes and manages the memtable; there is no user knob.

**Gotcha.** A write burst triggers many flushes, and the later compactions compete with serving traffic for CPU. This is why write-heavy tables need node headroom, not just storage.

Worked example: read path order.

```text
read row -> memtable (newest writes) -> block cache -> SSTable(s) in Colossus
```

Common mistake: quoting HBase `hbase.hregion.memstore.flush.size` as a Cloud Bigtable setting. It is a real HBase tuning parameter with no Bigtable equivalent.

## Does Bigtable provide automatic data expiration?

In [ ]:
# ILLUSTRATIVE - requires google: install to run, or read on.
from google.cloud.bigtable import column_family

# keep the newest 3 versions and drop anything older than 30 days
rule = column_family.UnionRule(
    column_family.MaxVersionsGCRule(3),
    column_family.MaxAgeGCAgeRule(30 * 24 * 60 * 60),
)
table.column_family("r", gc_rule=rule).create()

```bash
# inspect and change a family's GC policy from the CLI
cbt -project acme-prod -instance acme-bt ls device_readings
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxage=7d
```

**Main point.** Yes, through GC policy, not through a TTL column. A `max_age` rule on a column family expires cells after a duration, and compaction removes them.

**How it works.** Set `max_age` in seconds with the client library, or `maxage=30d` with cbt. You can combine it with `max_num_versions` using union or intersection. Expiration is per family, so hot data with a short life and reference data with no expiry can share a table.

**Gotcha.** Expiration is not immediate. A cell past its age is removed when compaction processes it, so reads may still see it briefly. Use a read-time timestamp filter when freshness must be exact.

Worked example: TTL on an events family.

```bash
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxage=7d
cbt -project acme-prod -instance acme-bt ls device_readings
```

Common mistake: storing the TTL as a value and writing a nightly delete job. Let the GC policy do the work; it needs no scanner and no downtime.

## Does Bigtable provide support for aggregations and analytics?

Bigtable is optimized for key-based reads and writes, not for scanning aggregates. In order of preference: maintain aggregates at write time (counter families or aggregate cells), use continuous materialized views, query with GoogleSQL for Bigtable, use Data Boost for large read jobs, or query the data from BigQuery. Hadoop or Dataflow is the heavy-lift path, not the first answer.

**Mental model.** Bigtable serves key lookups; it is not an analytics engine. Two managed paths bridge the gap: aggregate column families for write-time counters, and BigQuery federation for analytical reads.

**How it works.** Aggregate families maintain sum, min, max, and similar values as writes arrive, so you read a running total instead of scanning. BigQuery can query a Bigtable table through an external (federated) table, and Data Boost serves large reads without touching serving capacity.

**Gotcha.** Aggregates only cover the values you configured, and federation is for row-key-shaped reads, not arbitrary joins. Real analytics still belongs on a columnar copy.

Worked example: BigQuery over Bigtable.

```sql
CREATE EXTERNAL TABLE analytics.device_readings
OPTIONS (
  format = 'CLOUD_BIGTABLE',
  uris = ['https://googleapis.com/bigtable/projects/acme-prod/instances/acme-bt/tables/device_readings'],
  bigtable_options = '{"columnFamilies":[{"familyId":"r","columns":[{"qualifier":"temperature","type":"BYTES","encoding":"TEXT"}]}]}'
);

SELECT rowkey, temperature
FROM analytics.device_readings
WHERE rowkey LIKE 'acme#dev-00042%'
LIMIT 100;
```

Common mistake: running `GROUP BY` over a full federated table without a row-key filter. It reads everything and is slow and expensive; export to BigQuery instead.

## Can you explain how Bigtable handles storage and retrieval of large objects?

It does not. Bigtable has row and cell size limits and is optimized for many small rows, so large objects belong in Cloud Storage. Store the object there and keep a reference (bucket, path, size, checksum) in Bigtable.

**Mental model.** Bigtable can hold large cell values up to a service maximum on the order of 100 MiB, but it is not object storage. Large cells cost cache, slow reads, and travel uncompressed on the wire.

**How it works.** Store a small pointer in Bigtable and the bytes in Cloud Storage when objects exceed a few MiB. The row key gives you the fast metadata lookup; the object store serves the payload. Keep the pointer and the checksum in the cell.

**Gotcha.** A cell that is tens of megabytes evicts useful cache data and makes every read of that row expensive, even if you only need a different column.

Worked example: pointer pattern.

```text
row key:  media#img-7781
family m: gcs_uri = "gs://acme-media/img-7781.jpg"
          size    = "4123456"
          sha256  = "..."
payload:  stored in Cloud Storage, served by a signed URL
```

Common mistake: storing a whole image in a cell and then reading the row for a thumbnail. Fetch only the columns you need, and put the bytes elsewhere.

## Can you explain how Bigtable handles range scans and filters?

Filters are applied server-side, so a query only returns the cells it needs.

**Mental model.** A scan walks rows in key order over a half-open range `[start, end)`. Filters then drop rows, columns, or cells before they reach the client, but the server still reads the key range you asked for.

**How it works.** Push the prefix into the key range so the scan starts and stops at the right place. Use cell filters to cut columns and versions, and a row limit to bound the result. In Python, `read_rows(start_key=..., end_key=...)` plus `row_filters` does this.

**Gotcha.** A filter is not a seek. Filtering by a value or a non-key column scans the whole range and throws most of it away. Design the key so the range itself is narrow.

Worked example: narrow range plus filters.

```bash
cbt -project acme-prod -instance acme-bt read device_readings prefix=acme#dev-00042# count=10
```

```python
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"acme#dev-00042#",
    end_key=b"acme#dev-00042$",
    limit=10,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
```

Common mistake: scanning a whole table and filtering by `device_id` in the client. Put `device_id` in the key prefix so the server does the narrowing.

In [ ]:
# ILLUSTRATIVE - requires google: install to run, or read on.
from google.cloud.bigtable import row_filters

# review-only: read the newest version of a few columns in a narrow key range
rows = table.read_rows(
    start_key=b"acme#dev-00042#",
    end_key=b"acme#dev-00042$",
    limit=50,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
rows.consume_all()
for row in rows.rows:
    print(row.row_key.decode(), {c.qualifier.decode(): c.value.decode() for c in row.cells[0]})


## Can you explain how Bigtable handles large-scale data migration?

Bigtable provides tools and utilities to facilitate large-scale data migration, allowing you to import/export data efficiently.

**Mental model.** Migration is a bulk write problem. Read from the source, write to Bigtable with batched mutations, and shape the key so writes spread across tablets from the start.

**How it works.** Dataflow templates and the Spark connector do bulk import and export. For HBase, offline export followed by import, or online replication from HBase to Bigtable, are the documented paths. Write with your own timestamps if the source has them, and confirm the GC policy keeps them.

**Gotcha.** A migration that writes keys in source order can hotspot. Shuffle or salt the incoming stream, and watch node CPU so compaction keeps up with the import.

Worked example: migration stages.

```text
1. create table + families + GC policy (before importing)
2. export source to Parquet or HFiles
3. bulk import through Dataflow or the Spark connector, keys shuffled
4. verify row counts and checksums, then cut over reads
```

Common mistake: importing first and setting the GC policy after. An `max_age` rule added later can drop the imported history in one compaction.

## Can you explain the role of Bigtable's read-modify-write operation?

Bigtable's read-modify-write operation allows you to read data, modify it, and write it back atomically within a single transaction, ensuring consistency.

**Mental model.** Read-modify-write is an application pattern, not a server transaction. You read a row, compute a new value, and write it back. Bigtable gives you the tool to make it safe: a conditional mutation.

**How it works.** Put the expected current state in a predicate filter and the new state in the mutation. `CheckAndMutateRow` applies the write only if the predicate matches. If it does not, re-read and retry with backoff. For counters, use an aggregate family or a sharded counter to avoid contention.

**Gotcha.** An unconditional read-modify-write loses updates under concurrency. The wider the gap between the read and the write, the more likely the loss.

Worked example: safe update with a version guard.

```python
row = table.conditional_row(b"acct-7781")
row.set_cell("s", b"balance", str(new_balance).encode())
row.set_cell("s", b"version", str(version + 1).encode())
row.commit()
```

Common mistake: retrying the whole operation without re-reading. The predicate must compare against the value you actually read.

## Can you explain the role of Bigtable's client-side buffering and batching in optimizing write operations?

Bigtable's client-side buffering and batching allow you to group multiple write operations together before sending them to the server, reducing network overhead and improving write performance.

**Mental model.** Batching amortizes network and RPC overhead across many mutations. Buffering groups small writes so fewer, larger requests reach the service.

**How it works.** `mutate_rows` sends a batch of rows in one call. The client library batches within limits and retries failed entries. Larger batches raise throughput up to a point, then latency and memory use grow and a retry re-sends more work.

**Gotcha.** Bigger is not always better. A huge batch can time out, and a retry can re-apply mutations unless they are idempotent. Tune batch size against tail latency, not just throughput.

Worked example: batch by count and size.

```python
table.mutate_rows(rows)   # rows is a list; the library chunks the call
# keep each row small and each batch to a few hundred rows, then measure p99
```

Common mistake: buffering writes in the application for seconds to build a big batch. That adds latency and risks losing the buffer on a crash; the service already has a commit log.

## Tuning row keys for even load and locality

**Mental model.** Two goals pull in opposite directions. Even load wants writes spread across the whole key space. Locality wants the rows one query needs to sit next to each other. A good key satisfies both for the queries that matter, and accepts a trade-off for the rest.

**How it works.** Diagnose first. Key Visualizer shows per-prefix traffic; a bright horizontal band on a prefix means a hotspot. Then choose a fix: field promotion moves a high-cardinality field to the front, salting prepends a fixed-width bucket, hashing spreads a single hot key, and a reversed timestamp turns "latest first" into a prefix scan.

**Gotcha.** Salting destroys free range scans over the salted field, because the salt reorders rows. Add it only to the leading, hottest part of the key, and keep the rest sorted.

Worked example: read the symptom, then fix the key.

```text
symptom: writes all land on the last tablet, node CPU is pegged
cause:   key = 2026-10-04T12:00:00Z#device-42 (ascending timestamp first)
fix:     key = 07#device-42#9999999999 (16 salt buckets, reversed timestamp)

symptom: one user row takes most reads, one tablet is hot
cause:   key = CONFIG#global-flags
fix:     key = 01#CONFIG#global-flags plus an application cache
```

A hotspot check is simple arithmetic: compute the share of writes per key prefix and compare the largest share to an even split.

```python
from collections import Counter

counts = Counter(k.split("#")[0] for k in ["00#d1", "00#d2", "00#d3", "03#d4", "03#d5"])
total = sum(counts.values())
worst = max(counts.values()) / total
print(f"worst prefix share = {worst:.0%} across {len(counts)} prefixes")
```

Common mistake: salting and then querying the old unsalted prefix. Reads must use the same fan-out as writes.

## Tuning GC policies and version counts

**Mental model.** GC policy is a storage-versus-read-cost dial. More retained versions mean more bytes, more compaction work, and slower wide reads. Fewer versions mean cheaper storage and faster reads, but less history. Set the rule per column family from the retention the product actually needs.

**How it works.** `max_num_versions` caps versions per cell. `max_age` caps age. `union` keeps a cell if either rule keeps it, so it is the permissive combination. `intersection` keeps a cell only if both keep it, so it is the restrictive one. The rule is enforced at compaction.

**Gotcha.** The default policy is not a storage cap. A table that appends to the same cell every second can retain far more than expected until you set a version limit.

Worked example: choose per family, not per table.

```text
family profile: {"max_num_versions": 1}          # only current state, cheapest
family events:  {"max_age": "2592000s"}          # 30 days of history
family audits:  {"max_num_versions": 12}         # last 12 changes, no age cap
combined:       {"union": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
```

```bash
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings events maxage=30d
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings profile maxversions=1
cbt -project acme-prod -instance acme-bt ls device_readings
```

Common mistake: using `union` to try to cap storage. Union keeps cells that any rule keeps, so it retains more; use `intersection` to bound bytes.

## Tuning cluster sizing, nodes, and app profiles

**Mental model.** Nodes buy throughput; storage grows on its own. Size the cluster from peak requests per second, add headroom for compaction, and use app profiles to route that traffic to the right cluster with the right consistency.

**How it works.** Plan on the documented per-node ceiling of roughly 10,000 simple reads or writes per second, and target well below it (about 60 to 70 percent) so compaction and spikes have room. Autoscaling adjusts between a minimum and maximum against a target utilization. Choose SSD for latency-sensitive workloads. An app profile selects single-cluster routing (read-your-writes) or multi-cluster routing (nearest replica, eventual consistency), and can be limited to a cluster group.

**Gotcha.** Read and write load share the same nodes, so take the larger of the two estimates, not the sum. And size every cluster for full peak if you expect seamless failover.

Worked example: node count from peak throughput.

```text
peak reads  = 50,000/s   peak writes = 20,000/s
usable per node = 10,000 * 0.7 = 7,000/s
read nodes  = ceil(50,000 / 7,000) = 8
write nodes = ceil(20,000 / 7,000) = 3
cluster     = 8 nodes (the larger requirement)
```

```bash
cbt -project acme-prod -instance acme-bt createcluster acme-bt-eu \
  --zone=europe-west1-b --num-nodes=8
cbt -project acme-prod -instance acme-bt createappprofile acme-bt eu-serving \
  --route-any
cbt -project acme-prod -instance acme-bt createappprofile acme-bt ledger \
  --route-to=acme-bt-us-east1
```

Common mistake: sizing on average QPS. Peak-to-average ratios above three are normal in serving workloads.

In [ ]:
# ILLUSTRATIVE - requires google: install to run, or read on.
from google.cloud import bigtable
from google.cloud.bigtable import enums

# review-only: choose routing with an app profile, then tune nodes per cluster
admin = bigtable.Client(project="acme-prod", admin=True)
instance = admin.instance("acme-bt")

instance.app_profile(
    "eu-serving",
    routing_policy=enums.RoutingPolicyType.ANY,   # nearest healthy cluster
    description="latency-first reads",
).create()

instance.app_profile(
    "ledger",
    routing_policy=enums.RoutingPolicyType.SINGLE,
    description="read-your-writes on one cluster",
).create()


## Tuning reads and writes: batching, filters, and caching

**Mental model.** Reads are cheapest when they are point lookups or narrow key ranges that ask for few versions and few columns. Writes are cheapest when the key spreads them and GC bounds the versions they create. Everything else is overhead you can design away.

**How it works.** Bound every scan with a start key, an end key, and a limit. Use filters to cut columns and versions: a cells-per-column limit, a column-qualifier regex, a timestamp range. Batch writes with `mutate_rows`, but keep batches small enough to avoid timeouts and make retries idempotent. For very large read jobs, use Data Boost or a replica so serving traffic is untouched.

**Gotcha.** A row filter is applied after the server reads the key range. Filtering on a value does not narrow the scan; only the key range does.

Worked example: bound reads, then federate for analytics.

```python
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"acme#dev-00042#",
    end_key=b"acme#dev-00042$",
    limit=50,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
rows.consume_all()
```

For analytical questions, query Bigtable from BigQuery through an external table and keep a row-key filter so predicates push down.

```sql
SELECT rowkey, MAX(temperature) AS max_temp
FROM analytics.device_readings
WHERE rowkey LIKE 'acme#dev-00042%'
GROUP BY rowkey;
```

Common mistake: reading a wide row and filtering columns in the application. A qualifier filter drops those bytes before they cross the network.

## Things to remember: key numbers and limits


| Topic | Number / fact | Why it matters |
|---|---|---|
| Row key | max 4 KiB, byte-sorted across tablets | the row key IS the index; no secondary indexes |
| Row/cell size | design many small rows; a cell has a service maximum and oversized rows hurt latency and write amplification | unbounded rows per entity (e.g. one row per user, all events) is the classic scaling failure |
| Transactions | ACID at the single-row level only, including conditional read-modify-write | model aggregates and counters per row; cross-row atomicity does not exist |
| Consistency | single-cluster reads are strongly consistent; multi-cluster replication is eventually consistent | pick app profile routing per read consistency need |
| Versioning | each cell keeps multiple timestamped versions; GC policies bound age and count | versions are not time travel for analytics; GC them deliberately |
| Full-text / aggregations / auto-indexing | not provided | pair with Elasticsearch for search, BigQuery for analytics |
| Throughput | scales per node with autoscaling; hot rows do not split | a hot row key caps throughput no matter how many nodes |


## Things to remember: semantics to memorize


* **The row key decides everything.** Tablets split on key ranges, so a monotonically increasing key (timestamp-first) writes to one tablet at a time — hotspotting by design. Hash or reverse timestamps deliberately.
* **Reads are point-lookup or prefix scan.** Anything beyond that is a full-table scan: there is no query planner, no secondary index, no ad-hoc analytics.
* **GC policies are your TTL.** Age and version-count policies keep time-series rows bounded; without them every cell grows forever.
* **App profiles route traffic.** Single-cluster routing for strong consistency, multi-cluster for failover — the choice is per workload, not per table.
* **Compaction rewrites storage.** Write amplification during compactions is normal; watch latency during windows, and let GC policies shrink what needs re-writing.


## Things to remember: mistakes that keep coming back


* Timestamp-first row keys — all writes hit the newest tablet; salt, hash, or reverse the timestamp (`Long.MAX_VALUE - ts`).
* One unbounded row per entity — flip the time component into the key so a row stays one event.
* Expecting cross-row transactions or foreign keys — remodel around single-row atomicity, or use a relational store.
* Using Bigtable for ad-hoc analytics or search — it is neither; it is the high-throughput lookup layer beside BigQuery and Elasticsearch.
* Letting versions accumulate with no GC policy — storage and scan cost grow with every write.
* Sizing the cluster while hotspotting — more nodes cannot rescue a bad row key; fix the key first.
* Treating replicated clusters as strongly consistent — use a single-cluster app profile where reads must be current.


## Real world use cases for Bigtable

### 1. IoT telemetry serving

Millions of devices write readings every few seconds; the application shows the latest readings per device and a downsampled history.

In [ ]:
# ILLUSTRATIVE - requires google: install to run, or read on.
from google.cloud import bigtable
from google.cloud.bigtable import column_family

client = bigtable.Client(project="acme-prod", admin=True)
table = client.instance("acme-bt").table("device_readings")
table.create()

# expire readings automatically after 7 days
cf = table.column_family("r", gc_rule=column_family.MaxAgeGCAgeRule(7 * 24 * 60 * 60))
cf.create()

# device prefix plus a reversed timestamp so the newest reading sorts first
row = table.direct_row(b"dev-00042#2026-10-04T12:00:00Z")
row.set_cell("r", b"temperature", b"21.4")
row.set_cell("r", b"humidity", b"48")
row.commit()

```bash
# the same operations from the cbt CLI
cbt -project acme-prod -instance acme-bt createtable device_readings
cbt -project acme-prod -instance acme-bt createfamily device_readings r
cbt -project acme-prod -instance acme-bt set device_readings dev-00042#2026-10-04T12:00:00Z r:temperature=21.4
cbt -project acme-prod -instance acme-bt read device_readings prefix=dev-00042#
```

Why Bigtable: millisecond key lookups at very high write rates, with TTL enforced by garbage collection instead of a cleanup job. Salt or bucket the key prefix when one device or tenant dominates traffic.

### 2. User activity and message history

A messaging product stores per-conversation history and a per-user inbox. Reads are prefix scans ordered by time.

In [ ]:
# ILLUSTRATIVE - requires google: install to run, or read on.
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"conv-7781#",     # newest first because the timestamp is reversed in the key
    end_key=b"conv-7781$",
    limit=50,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
rows.consume_all()

Why Bigtable: a sorted row key gives "the last 50 messages" with a range scan and no secondary index, while appends stay cheap. Model the "all conversations for a user" query as a second row-key layout or an asynchronous secondary index.

### 3. Online feature store for ML serving

A fraud model needs 30-day aggregates per card at prediction time.

```text
row key: card#c-9182#feature#velocity_24h
columns: f:value, f:updated_at
write:   conditional mutation so an older write cannot overwrite a newer value
read:    one row per feature at scoring time (sub-10 ms p99)
train:   export or stream the same features to BigQuery for offline training
```

Why Bigtable: the serving path needs predictable latency at high QPS, which is exactly a row-key lookup. Do not scan Bigtable to build training sets - use BigQuery or Parquet, and keep both paths fed by one feature definition to avoid training/serving skew.

### 4. Real-time bidding and profile lookup

Ad tech keeps a user profile that every bid request reads, and reads must be fast in every region.

```text
row key:  bucket#user_id          (bucket spreads hot users across tablets)
families: profile (small, rarely changing), events (append-only, GC by age)
routing:  multi-cluster app profile so bids read the nearest replica
```

Why Bigtable: high read throughput with replicated low-latency reads. Multi-cluster routing accepts eventual consistency for profile updates, which bidding tolerates, in exchange for serving reads close to the user.

### 5. Operational serving plus BigQuery analytics

The same data serves the application and the analyst. Change Streams feed the analytical copy instead of a dual write.

```text
Bigtable (serving) --Change Streams--> Dataflow --> BigQuery (analytics) --> BI
                          |
                   Data Boost for large read jobs without competing with serving traffic
```

```sql
-- analytical questions run in BigQuery, not against the serving table
SELECT DATE(event_ts) AS ds,
       COUNT(*) AS events,
       APPROX_QUANTILES(latency_ms, 100)[OFFSET(95)] AS p95
FROM analytics.events_enriched
GROUP BY ds
ORDER BY ds;
```

Why Bigtable: the serving table stays optimized for key lookups, analytics runs on a columnar copy, Change Streams remove the fragile dual write, and Data Boost isolates heavy scans from production traffic.

**Mental model.** Work backward from the access pattern. Write down the top reads and writes, their latency targets, and their request rates. If the answers are "by key" or "by a narrow ordered range" at high rate, Bigtable fits.

**How it works.** The five cases below all use a key-first design, per-family GC for different retention, and app profiles for routing or read isolation. Each one pairs Bigtable with an analytical copy so heavy scans never touch the serving path.

**Gotcha.** A use case is not a schema. Two teams can both say "IoT telemetry" and need different keys, different families, and different GC rules, depending on whether they query by device, by site, or by time window.

Common mistake: copying a case study's key design without its query list. The key is a function of the queries, not of the industry.

The pattern to reuse across all five cases:

```text
serving reads and writes  -> Bigtable, key-first, per-family GC
analytics and dashboards   -> BigQuery, fed by Change Streams or export
large read jobs            -> Data Boost or a replica, never the serving path
routing and consistency    -> app profiles, one per workload
```

## Bigtable exercises

These exercises use only the Python standard library and model Bigtable key, sizing, and GC decisions with small runnable calculations. Each one states the task, gives a hint, and names the expected result before the code cell. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are engineering arithmetic, not benchmarks: bytes and versions for storage, ratios for hotspot and selectivity checks. Treat the asserted values as a check on the formula.

### Try it 1. Design a row key from access patterns

**Task.** Build a row key for the queries "latest reading for one device, newest first" and "all readings for a device in a time window". The key must be `salt#device_id#reverse_ts` with a two-digit salt, and the reversed timestamp must sort newest first.

**Hint.** Use a far-future sentinel and subtract the epoch milliseconds: `9_999_999_999_999 - ts_ms`. Format with `%013d`, and derive the salt with `hashlib.md5(device_id.encode())` modulo the bucket count.

**Expected result.** Every key starts with a two-digit salt below 16, keys for one device sort newest first, and a prefix scan returns them in that order.

In [ ]:
import hashlib

BUCKETS = 16
MAX_TS = 9_999_999_999_999


def row_key(device_id, ts_ms):
    salt = int(hashlib.md5(device_id.encode()).hexdigest(), 16) % BUCKETS
    return f"{salt:02d}#{device_id}#{MAX_TS - ts_ms:013d}"


base = 1_700_000_000_000
timestamps = [base + offset for offset in (120_000, 60_000, 0)]  # newest first
keys = [row_key("dev-00042", ts) for ts in timestamps]
for key in keys:
    print(key)

salts = [key.split("#")[0] for key in keys]
print("distinct salts:", sorted(set(salts)))
print("newest first:", keys == sorted(keys))

assert all(len(salt) == 2 and 0 <= int(salt) < BUCKETS for salt in salts)
assert keys == sorted(keys)
assert keys[0].endswith(f"{MAX_TS - (base + 120_000):013d}")
assert len(set(keys)) == 3
print("ok")


### Try it 2. Compute storage growth from rows, versions, and cell size

**Task.** A table has 500 million rows, an average key plus value size of 120 bytes, keeps 3 versions per cell, and replicates to 2 clusters. Compute raw stored bytes, the GiB figure, and the bytes after a 3x compression ratio.

**Hint.** Raw bytes are `rows * cell_bytes * versions * clusters`. Compression divides by the ratio. Use `1024 ** 3` for GiB and round only when printing.

**Expected result.** `360,000,000,000` raw bytes, about `335.3 GiB`, and `120,000,000,000` compressed bytes.

In [ ]:
ROWS = 500_000_000
CELL_BYTES = 120
VERSIONS = 3
CLUSTERS = 2
COMPRESSION_RATIO = 3.0

raw_bytes = int(ROWS * CELL_BYTES * VERSIONS * CLUSTERS)
compressed_bytes = int(raw_bytes / COMPRESSION_RATIO)

print(f"raw bytes        = {raw_bytes:,}")
print(f"raw GiB          = {raw_bytes / 1024 ** 3:.1f}")
print(f"compressed bytes = {compressed_bytes:,}")

assert raw_bytes == 360_000_000_000
assert round(raw_bytes / 1024 ** 3, 1) == 335.3
assert compressed_bytes == 120_000_000_000
print("ok")


### Try it 3. Estimate node count from throughput targets

**Task.** Peak load is 50,000 reads per second and 20,000 writes per second. A node handles about 10,000 simple reads or writes per second, and you plan to use 70 percent of that. Compute the nodes needed for reads, for writes, and the cluster size.

**Hint.** Use `math.ceil` on `peak / (per_node * target_utilisation)`. Reads and writes share nodes, so the cluster needs the larger of the two numbers, not the sum.

**Expected result.** 8 read nodes, 3 write nodes, and a cluster of 8 nodes.

In [ ]:
import math

PEAK_READS = 50_000
PEAK_WRITES = 20_000
PER_NODE = 10_000
TARGET = 0.7

usable_per_node = PER_NODE * TARGET
read_nodes = math.ceil(PEAK_READS / usable_per_node)
write_nodes = math.ceil(PEAK_WRITES / usable_per_node)
cluster_nodes = max(read_nodes, write_nodes)

print(f"usable per node = {usable_per_node:,.0f} operations/s")
print(f"read nodes      = {read_nodes}")
print(f"write nodes     = {write_nodes}")
print(f"cluster nodes   = {cluster_nodes}")

assert usable_per_node == 7_000.0
assert read_nodes == 8
assert write_nodes == 3
assert cluster_nodes == 8
print("ok")


### Try it 4. Size a scan from row count and filter selectivity

**Task.** A range scan covers 8 million rows of 200 bytes each. A qualifier filter keeps 1 column in 40, and a version filter keeps 1 version in 5. Compute rows returned, bytes read before filters, and bytes returned after filters.

**Hint.** Rows returned is `rows * column_selectivity * version_selectivity`. Bytes before filters use the full row count; bytes after use the returned rows. Use integer division only when the arithmetic is exact.

**Expected result.** 40,000 rows returned, `1,600,000,000` bytes read before filters, and `8,000,000` bytes returned.

In [ ]:
ROWS = 8_000_000
ROW_BYTES = 200
COLUMN_SELECTIVITY = 1 / 40
VERSION_SELECTIVITY = 1 / 5

rows_returned = int(ROWS * COLUMN_SELECTIVITY * VERSION_SELECTIVITY)
bytes_before = ROWS * ROW_BYTES
bytes_after = rows_returned * ROW_BYTES
reduction = bytes_before // bytes_after

print(f"rows scanned    = {ROWS:,}")
print(f"rows returned   = {rows_returned:,}")
print(f"bytes before    = {bytes_before:,}")
print(f"bytes returned  = {bytes_after:,}")
print(f"reduction       = {reduction}x")

assert rows_returned == 40_000
assert bytes_before == 1_600_000_000
assert bytes_after == 8_000_000
assert reduction == 200
print("ok")


### Try it 5. Detect a hotspot from key-prefix distribution

**Task.** Given request counts per key prefix, compute the total, the even share, the busiest prefix, its ratio to the even share, and the prefixes above twice the even share.

**Hint.** The even share is `total / len(counts)`. The ratio is `count / even_share`. List comprehension over the counts finds the offenders.

**Expected result.** Total `134,500`, even share `16,812.5`, busiest `00` at about `2.97x`, and `["00"]` above the 2.0x threshold.

In [ ]:
counts = {
    "00": 50_000,
    "01": 20_000,
    "02": 18_000,
    "03": 12_000,
    "04": 10_000,
    "05": 9_000,
    "06": 8_000,
    "07": 7_500,
}

total = sum(counts.values())
even_share = total / len(counts)
ratios = {prefix: count / even_share for prefix, count in counts.items()}
busiest = max(ratios, key=ratios.get)
hot = [prefix for prefix, ratio in ratios.items() if ratio > 2.0]

print(f"total requests = {total:,}")
print(f"even share     = {even_share:,.1f}")
print(f"busiest        = {busiest} at {ratios[busiest]:.2f}x")
print("prefixes above 2.0x:", hot)

assert total == 134_500
assert even_share == 16_812.5
assert round(ratios[busiest], 2) == 2.97
assert hot == ["00"]
print("ok")


### Try it 6. Plan a GC policy from retention days and versions

**Task.** A family must keep 30 days of history but never more than 5 versions per cell. Write a function that classifies a proposed policy as `union`, `intersection`, or `single`, and return the tighter combination for the requirement.

**Hint.** `union` keeps a cell if any rule keeps it. `intersection` keeps it only if all rules keep it, which is what "no more than 5 versions AND no older than 30 days" means. Convert days to seconds with `days * 86_400`.

**Expected result.** The combined policy is `intersection`, `max_age_seconds` is `2,592,000`, and the version cap is `5`.

In [ ]:
def plan_gc(retention_days, max_versions):
    if retention_days and max_versions:
        policy = "intersection"
    elif retention_days or max_versions:
        policy = "single"
    else:
        policy = "none"
    return {
        "policy": policy,
        "max_age_seconds": retention_days * 86_400 if retention_days else None,
        "max_num_versions": max_versions,
    }


plan = plan_gc(retention_days=30, max_versions=5)
print(plan)

assert plan["policy"] == "intersection"
assert plan["max_age_seconds"] == 2_592_000
assert plan["max_num_versions"] == 5
assert plan_gc(0, 3)["policy"] == "single"
assert plan_gc(7, 0)["max_age_seconds"] == 604_800
assert plan_gc(0, 0)["policy"] == "none"
print("ok")


### Try it 7. Model salting fan-out

**Task.** A single hot key takes 64,000 writes per second. Model fan-out with 4, 8, and 16 salt buckets, and pick the smallest bucket count that keeps each salted key under 10,000 writes per second.

**Hint.** Writes per bucket are `total / buckets`. Iterate the candidate bucket counts in ascending order and stop at the first that satisfies the limit.

**Expected result.** 16,000 then 8,000 then 4,000 writes per bucket, and a chosen fan-out of 8 buckets.

In [ ]:
TOTAL_WRITES = 64_000
PER_KEY_LIMIT = 10_000

fan_out = {}
for buckets in (4, 8, 16):
    per_key = TOTAL_WRITES / buckets
    fan_out[buckets] = per_key
    print(f"{buckets:>2} buckets -> {per_key:,.0f} writes/s per salted key")

chosen = next(b for b in (4, 8, 16) if TOTAL_WRITES / b <= PER_KEY_LIMIT)
print("chosen fan-out:", chosen)

assert fan_out[4] == 16_000.0
assert fan_out[8] == 8_000.0
assert fan_out[16] == 4_000.0
assert chosen == 8
print("ok")


### Try it 8. Plan a backup schedule from a recovery point objective

**Task.** A table receives 2,000 writes per second. The recovery point objective allows losing at most 15 minutes of data, and a backup takes 4 minutes. Decide whether a 10-minute backup interval meets the objective, and compute the worst-case data loss.

**Hint.** Worst-case loss is `(interval + backup_duration)` minutes of writes, because a crash during a backup can lose the interval plus the in-progress backup window. Compare that to the 15-minute objective in seconds.

**Expected result.** Worst-case loss is 14 minutes (`1,680,000` writes), a 10-minute interval meets the objective, and a 15-minute interval does not.

In [ ]:
WRITES_PER_SECOND = 2_000
RPO_MINUTES = 15
BACKUP_MINUTES = 4


def worst_case_loss_minutes(interval_minutes):
    return interval_minutes + BACKUP_MINUTES


loss_10 = worst_case_loss_minutes(10)
loss_15 = worst_case_loss_minutes(15)
loss_10_writes = loss_10 * 60 * WRITES_PER_SECOND

print(f"10-minute interval -> {loss_10} minutes, {loss_10_writes:,} writes worst case")
print(f"15-minute interval -> {loss_15} minutes worst case")
print("10-minute interval meets RPO:", loss_10 <= RPO_MINUTES)

assert loss_10 == 14
assert loss_10_writes == 1_680_000
assert loss_10 <= RPO_MINUTES
assert loss_15 > RPO_MINUTES
print("ok")
